# Modul 1: Dekomposisi Sinyal Deret Waktu Kelistrikan
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 1 (Minggu 1):** Mengidentifikasi komponen tren ($T_t$), musiman ($S_t$), siklis ($C_t$), dan residu ($R_t$) pada data beban listrik secara komputasional.

Model dekomposisi klasik:
- **Aditif:** $y_t = T_t + S_t + R_t$ (amplitudo musiman relatif konstan)
- **Multiplikatif:** $y_t = T_t \times S_t \times R_t$ (amplitudo musiman membesar seiring tren)

In [ ]:
# Jika dijalankan di Google Colab, pustaka inti umumnya sudah terpasang.
# !pip install -q pandas numpy matplotlib statsmodels

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True

## 1. Memuat Dataset Beban Listrik Per-Jam
Notebook ini otomatis membaca `hourlyloadpln.csv` jika tersedia secara lokal, atau membangkitkan profil beban gardu induk realistis (30 hari, interval 1 jam) saat dijalankan mandiri di Google Colab.

In [ ]:
local_csv = Path('../latihanbab1/hourlyloadpln.csv')
if local_csv.exists():
    df_raw = pd.read_csv(local_csv).dropna(how='all', axis=1)
    y = pd.Series(df_raw['GroundTruth'].values, name='load_mw')
    y.index = pd.date_range('2026-01-01', periods=len(y), freq='h')
    print(f"Dataset lokal dimuat: {len(y)} titik data.")
else:
    rng = np.random.default_rng(42)
    n_hours = 24 * 30  # 30 hari
    idx = pd.date_range('2026-01-01', periods=n_hours, freq='h')
    t = np.arange(n_hours)
    hour = idx.hour
    trend = 420 + 0.08 * t
    daily = 95 * np.sin(2 * np.pi * (hour - 6) / 24) + 55 * np.exp(-((hour - 19) ** 2) / 8)
    weekly = -25 * (idx.dayofweek >= 5).astype(float)
    noise = rng.normal(0, 10, n_hours)
    y = pd.Series(trend + daily + weekly + noise, index=idx, name='load_mw')
    print(f"Dataset simulasi PLN dibangkitkan: {len(y)} titik data.")

y.head()

## 2. Dekomposisi Aditif vs. Multiplikatif (Periode Harian $m = 24$)

In [ ]:
decomp_add = seasonal_decompose(y, model='additive', period=24)
decomp_mul = seasonal_decompose(y, model='multiplicative', period=24)

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
decomp_add.observed.plot(ax=axes[0], color='navy', title='Observed Load (MW)')
decomp_add.trend.plot(ax=axes[1], color='darkred', title='Trend Component (CMA-24)')
decomp_add.seasonal.plot(ax=axes[2], color='darkgreen', title='Daily Seasonality (m=24)')
decomp_add.resid.plot(ax=axes[3], color='black', title='Residual / Irregular Component')
plt.tight_layout()
plt.show()

print("Statistik Residual Aditif:")
print(f"  Mean = {decomp_add.resid.mean():.4f} MW | Var = {decomp_add.resid.var():.4f}")
print("Statistik Residual Multiplikatif:")
print(f"  Mean = {decomp_mul.resid.mean():.4f} | Var = {decomp_mul.resid.var():.6f}")

## 3. Tugas & Pertanyaan Analisis (C3–C6)
1. **[C3]** Bandingkan plot residual model **aditif** dan **multiplikatif** di atas. Apakah nilai rata-rata residual aditif mendekati 0 dan multiplikatif mendekati 1?
2. **[C4]** Ubah periode dekomposisi menjadi `period=168` (siklus mingguan). Bandingkan pola musiman hari kerja vs. akhir pekan!
3. **[C5–C6]** Modifikasi amplitudo sinyal simulasi agar membesar proporsional terhadap tren (`daily * (1 + 0.001 * t)`), lalu buktikan secara kuantitatif model mana yang menghasilkan residual lebih homoskedastik!